In [1]:
%pip install lightgbm xgboost scikit-learn imbalanced-learn numpy pandas


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load


과제에서 활용하는 데이터를 Kaggle API 활용 또는 다운로드 후 불러오기를 통해서 다음과 같은 변수에 저장:
* 훈련 레이블 데이터: **TRAIN_LABEL**
* 테스트 레이블 데이터 (id, pid, timestamp 포함): **TEST_DATA**

센서별 데이터는 별도 변수로 로드합니다.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

# ── Labels ────────────────────────────────────────────────────────────
train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

# ── Train sensors ─────────────────────────────────────────────────────
trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

# ── Test sensors ──────────────────────────────────────────────────────
testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

# Template required variables
TRAIN_LABEL = train_labels
TEST_DATA   = test_labels

# Ensure timestamps are numeric
for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)
print('Train PIDs:', sorted(train_labels.pid.unique()))
print('Test  PIDs:', sorted(test_labels.pid.unique()))

Train labels: (1456, 4) | Test labels: (1496, 4)
Train PIDs: ['01Z2', '70N8', '7PF3', 'CQ2G', 'D1XP', 'DT5C', 'F1ZM', 'LIUY', 'SE4Q', 'TPQI', 'Y21H']
Test  PIDs: ['13P2', '2XO3', '43JW', 'C8Q6', 'HDS9', 'NQRB', 'OL6N', 'P4DZ', 'QEYR', 'SNG7', 'TF0Y', 'WZDL']


## Step 1 — Per-Subject Z-Score Normalization

In [3]:
def subject_zscore(df, val_cols, pid_col='pid'):
    """Apply per-subject z-score normalization in-place.
    Subjects with zero std (constant signal) are left as-is.
    """
    df = df.copy()
    for col in val_cols:
        df[col] = df.groupby(pid_col)[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8)
        )
    return df

# Normalize single-value sensors
trainhr_n   = subject_zscore(trainhr,   ['value'])
traineda_n  = subject_zscore(traineda,  ['value'])
traintemp_n = subject_zscore(traintemp, ['value'])
trainibi_n  = subject_zscore(trainibi,  ['value'])
trainbvp_n  = subject_zscore(trainbvp,  ['value'])

testhr_n    = subject_zscore(testhr,    ['value'])
testeda_n   = subject_zscore(testeda,   ['value'])
testtemp_n  = subject_zscore(testtemp,  ['value'])
testibi_n   = subject_zscore(testibi,   ['value'])
testbvp_n   = subject_zscore(testbvp,   ['value'])

# ACC magnitude + normalize
for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

trainacc_n = subject_zscore(trainacc, ['magnitude'])
testacc_n  = subject_zscore(testacc,  ['magnitude'])

# EEG log1p transform (before normalization; raw values are heavily skewed)
EEG_COLS = ['delta', 'theta', 'lowAlpha', 'highAlpha',
            'lowBeta', 'highBeta', 'lowGamma', 'middleGamma']

for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

trainbrain_n = subject_zscore(trainbrain, EEG_COLS)
testbrain_n  = subject_zscore(testbrain,  EEG_COLS)

print('Normalization done.')

Normalization done.


## Step 2 — Window Feature Extraction

In [4]:
HALF_WIN = 2500   # ±2.5 seconds in milliseconds


def window_feats_single(sensor_df, label_df, sensor_name, half_win=HALF_WIN):
    """Extract statistical features from a ±half_win window around each label.
    sensor_df must have columns: pid, timestamp, value
    Returns DataFrame indexed by label row order.
    """
    sensor_df = sensor_df.sort_values(['pid', 'timestamp'])
    records = []

    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        s = sensor_df[sensor_df.pid == pid]
        win = s[(s.timestamp >= ts - half_win) &
                (s.timestamp <  ts + half_win)]['value']

        feat = {}
        n = len(win)
        if n >= 2:
            feat[f'{sensor_name}_mean']  = win.mean()
            feat[f'{sensor_name}_std']   = win.std()
            feat[f'{sensor_name}_min']   = win.min()
            feat[f'{sensor_name}_max']   = win.max()
            feat[f'{sensor_name}_range'] = win.max() - win.min()
            # Linear slope (trend direction within window)
            t_idx = np.arange(n)
            feat[f'{sensor_name}_slope'] = np.polyfit(t_idx, win.values, 1)[0]
        else:
            for suf in ['mean', 'std', 'min', 'max', 'range', 'slope']:
                feat[f'{sensor_name}_{suf}'] = np.nan

        # EDA-specific: artifact flag (fraction of zeros)
        if sensor_name == 'eda':
            raw_win = sensor_df[
                (sensor_df.pid == pid) &
                (sensor_df.timestamp >= ts - half_win) &
                (sensor_df.timestamp <  ts + half_win)
            ]['value']  # use raw (un-normalized) zeros? We already normalized, so
            # use the original traineda/testeda which haven't been modified in place
            feat['eda_valid'] = 1  # placeholder; computed separately below

        records.append(feat)

    return pd.DataFrame(records)


def eda_artifact_flag(raw_eda_df, label_df, half_win=HALF_WIN):
    """Returns eda_valid (1=good, 0=mostly zeros) and eda_zero_ratio per window."""
    raw_eda_df = raw_eda_df.sort_values(['pid', 'timestamp'])
    flags, ratios = [], []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        s = raw_eda_df[raw_eda_df.pid == pid]
        win = s[(s.timestamp >= ts - half_win) &
                (s.timestamp <  ts + half_win)]['value']
        if len(win) == 0:
            flags.append(0); ratios.append(1.0)
        else:
            zr = (win == 0).mean()
            ratios.append(zr)
            flags.append(0 if zr > 0.5 else 1)
    return pd.DataFrame({'eda_valid': flags, 'eda_zero_ratio': ratios})


def window_feats_ibi(ibi_df, label_df, half_win=HALF_WIN):
    """IBI: mean, std, RMSSD (HRV proxy). NaN-tolerant."""
    ibi_df = ibi_df.sort_values(['pid', 'timestamp'])
    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        s = ibi_df[ibi_df.pid == pid]
        win = s[(s.timestamp >= ts - half_win) &
                (s.timestamp <  ts + half_win)]['value'].values
        feat = {}
        if len(win) >= 2:
            feat['ibi_mean']  = np.mean(win)
            feat['ibi_std']   = np.std(win)
            feat['ibi_range'] = np.max(win) - np.min(win)
            diffs = np.diff(win)
            feat['ibi_rmssd'] = np.sqrt(np.mean(diffs**2)) if len(diffs) > 0 else np.nan
        else:
            for k in ['ibi_mean', 'ibi_std', 'ibi_range', 'ibi_rmssd']:
                feat[k] = np.nan
        records.append(feat)
    return pd.DataFrame(records)


def window_feats_acc(acc_df, label_df, half_win=HALF_WIN):
    """ACC: magnitude mean, std, energy."""
    acc_df = acc_df.sort_values(['pid', 'timestamp'])
    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        s = acc_df[acc_df.pid == pid]
        win = s[(s.timestamp >= ts - half_win) &
                (s.timestamp <  ts + half_win)]['magnitude'].values
        feat = {}
        if len(win) >= 5:
            feat['acc_mean']   = np.mean(win)
            feat['acc_std']    = np.std(win)
            feat['acc_energy'] = np.mean(win**2)
        else:
            feat['acc_mean'] = feat['acc_std'] = feat['acc_energy'] = np.nan
        records.append(feat)
    return pd.DataFrame(records)


def window_feats_eeg(brain_df, label_df, half_win=HALF_WIN):
    """EEG: mean per band + cognitive ratio features."""
    brain_df = brain_df.sort_values(['pid', 'timestamp'])
    eeg_cols = ['delta', 'theta', 'lowAlpha', 'highAlpha',
                'lowBeta', 'highBeta', 'lowGamma', 'middleGamma']
    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        s = brain_df[brain_df.pid == pid]
        win = s[(s.timestamp >= ts - half_win) &
                (s.timestamp <  ts + half_win)]
        feat = {}
        if len(win) >= 1:
            for col in eeg_cols:
                feat[f'eeg_{col}'] = win[col].mean()
            # Ratio features (cognitive load / arousal proxies)
            eps = 1e-8
            feat['eeg_theta_alpha'] = feat['eeg_theta'] / (feat['eeg_lowAlpha'] + feat['eeg_highAlpha'] + eps)
            feat['eeg_beta_alpha']  = (feat['eeg_lowBeta'] + feat['eeg_highBeta']) / (feat['eeg_lowAlpha'] + feat['eeg_highAlpha'] + eps)
            feat['eeg_hbeta_lgamma']= feat['eeg_highBeta'] / (feat['eeg_lowGamma'] + eps)
        else:
            for col in eeg_cols:
                feat[f'eeg_{col}'] = np.nan
            feat['eeg_theta_alpha'] = np.nan
            feat['eeg_beta_alpha']  = np.nan
            feat['eeg_hbeta_lgamma']= np.nan
        records.append(feat)
    return pd.DataFrame(records)


print('Feature extraction functions defined.')

Feature extraction functions defined.


In [5]:
print('Extracting train features (this may take 1-2 min)...')

tr_hr    = window_feats_single(trainhr_n,   train_labels, 'hr')
tr_eda   = window_feats_single(traineda_n,  train_labels, 'eda')
tr_temp  = window_feats_single(traintemp_n, train_labels, 'temp')
tr_ibi   = window_feats_ibi(trainibi_n,     train_labels)
tr_acc   = window_feats_acc(trainacc_n,     train_labels)
tr_eeg   = window_feats_eeg(trainbrain_n,   train_labels)
tr_eda_flag = eda_artifact_flag(traineda,   train_labels)
# Drop placeholder eda_valid column from single extractor, use real flag
tr_eda = tr_eda.drop(columns=['eda_valid'], errors='ignore')

train_feats = pd.concat([
    train_labels[['id', 'pid', 'timestamp', 'arousal']].reset_index(drop=True),
    tr_hr.reset_index(drop=True),
    tr_eda.reset_index(drop=True),
    tr_temp.reset_index(drop=True),
    tr_ibi.reset_index(drop=True),
    tr_acc.reset_index(drop=True),
    tr_eeg.reset_index(drop=True),
    tr_eda_flag.reset_index(drop=True),
], axis=1)

print('Train feature matrix:', train_feats.shape)
print('NaN counts (top 10):')
print(train_feats.isnull().sum().sort_values(ascending=False).head(10))

Extracting train features (this may take 1-2 min)...
Train feature matrix: (1456, 42)
NaN counts (top 10):
ibi_mean           812
ibi_std            812
ibi_range          812
ibi_rmssd          812
eeg_middleGamma    104
eeg_theta          104
eeg_beta_alpha     104
eeg_delta          104
eeg_lowAlpha       104
eeg_highAlpha      104
dtype: int64


In [6]:
print('Extracting test features...')

te_hr    = window_feats_single(testhr_n,   test_labels, 'hr')
te_eda   = window_feats_single(testeda_n,  test_labels, 'eda')
te_temp  = window_feats_single(testtemp_n, test_labels, 'temp')
te_ibi   = window_feats_ibi(testibi_n,     test_labels)
te_acc   = window_feats_acc(testacc_n,     test_labels)
te_eeg   = window_feats_eeg(testbrain_n,   test_labels)
te_eda_flag = eda_artifact_flag(testeda,   test_labels)
te_eda = te_eda.drop(columns=['eda_valid'], errors='ignore')

test_feats = pd.concat([
    test_labels[['id', 'pid', 'timestamp']].reset_index(drop=True),
    te_hr.reset_index(drop=True),
    te_eda.reset_index(drop=True),
    te_temp.reset_index(drop=True),
    te_ibi.reset_index(drop=True),
    te_acc.reset_index(drop=True),
    te_eeg.reset_index(drop=True),
    te_eda_flag.reset_index(drop=True),
], axis=1)

print('Test feature matrix:', test_feats.shape)

Extracting test features...
Test feature matrix: (1496, 41)


## Step 3 — Interaction Features

In [7]:
def add_interaction_features(df):
    """Add physiologically-motivated interaction and relative features."""
    df = df.copy()
    eps = 1e-8

    # Sympathetic activation composite
    if 'hr_mean' in df.columns and 'eda_mean' in df.columns:
        df['hr_eda_product'] = df['hr_mean'] * df['eda_mean']

    # Normalized HRV: IBI variability relative to mean IBI
    if 'ibi_std' in df.columns and 'ibi_mean' in df.columns:
        df['ibi_cv'] = df['ibi_std'] / (df['ibi_mean'].abs() + eps)

    # HR-IBI inverse relationship
    if 'hr_mean' in df.columns and 'ibi_mean' in df.columns:
        df['hr_ibi_ratio'] = df['hr_mean'] / (df['ibi_mean'].abs() + eps)

    # EDA * valid flag: zero out artifact windows
    if 'eda_mean' in df.columns and 'eda_valid' in df.columns:
        df['eda_mean_clean'] = df['eda_mean'] * df['eda_valid']
        df['eda_std_clean']  = df['eda_std']  * df['eda_valid']

    return df


train_feats = add_interaction_features(train_feats)
test_feats  = add_interaction_features(test_feats)

# Final feature columns (exclude id/pid/timestamp/arousal)
FEAT_COLS = [c for c in train_feats.columns
             if c not in ['id', 'pid', 'timestamp', 'arousal']]

print(f'Total features: {len(FEAT_COLS)}')
print(FEAT_COLS)

Total features: 43
['hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_range', 'hr_slope', 'eda_mean', 'eda_std', 'eda_min', 'eda_max', 'eda_range', 'eda_slope', 'temp_mean', 'temp_std', 'temp_min', 'temp_max', 'temp_range', 'temp_slope', 'ibi_mean', 'ibi_std', 'ibi_range', 'ibi_rmssd', 'acc_mean', 'acc_std', 'acc_energy', 'eeg_delta', 'eeg_theta', 'eeg_lowAlpha', 'eeg_highAlpha', 'eeg_lowBeta', 'eeg_highBeta', 'eeg_lowGamma', 'eeg_middleGamma', 'eeg_theta_alpha', 'eeg_beta_alpha', 'eeg_hbeta_lgamma', 'eda_valid', 'eda_zero_ratio', 'hr_eda_product', 'ibi_cv', 'hr_ibi_ratio', 'eda_mean_clean', 'eda_std_clean']


## Step 4 — LOSO CV with LightGBM + XGBoost

In [8]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

TRAIN_PIDS = sorted(train_feats['pid'].unique())
print(f'LOSO folds: {len(TRAIN_PIDS)} subjects')

X_all = train_feats[FEAT_COLS].values.astype(np.float32)
y_all = (train_feats['arousal'].values - 1).astype(int)   # 0-indexed for both models
pids  = train_feats['pid'].values

# Storage for OOF predictions
oof_lgb = np.zeros((len(train_feats), 5), dtype=np.float32)
oof_xgb = np.zeros((len(train_feats), 5), dtype=np.float32)

# Storage for test predictions (average across folds)
X_test = test_feats[FEAT_COLS].values.astype(np.float32)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float32)
test_xgb = np.zeros((len(test_feats), 5), dtype=np.float32)

# ── LightGBM params (conservative to avoid overfitting on ~10 subjects) ──
lgb_params = {
    'objective':        'multiclass',
    'num_class':        5,
    'metric':           'multi_logloss',
    'num_leaves':       31,
    'learning_rate':    0.05,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'min_child_samples':20,
    'lambda_l1':        0.1,
    'lambda_l2':        0.1,
    'verbose':         -1,
    'seed':             42,
    'n_jobs':          -1,
}

# ── XGBoost params ────────────────────────────────────────────────────
xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        5,
    'eval_metric':      'mlogloss',
    'max_depth':        4,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.7,
    'min_child_weight': 20,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'seed':             42,
    'verbosity':        0,
    'nthread':         -1,
}

loso_scores_lgb = []
loso_scores_xgb = []

for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid

    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]

    # Class-balanced sample weights
    sw_tr = compute_sample_weight('balanced', y_tr)

    # ── LightGBM ──────────────────────────────────────────────────────
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr,
                             feature_name=FEAT_COLS)
    dval_lgb   = lgb.Dataset(X_va, label=y_va, reference=dtrain_lgb)

    model_lgb = lgb.train(
        lgb_params,
        dtrain_lgb,
        num_boost_round=800,
        valid_sets=[dval_lgb],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )

    prob_va_lgb = model_lgb.predict(X_va)
    pred_va_lgb = prob_va_lgb.argmax(axis=1)
    ba_lgb = balanced_accuracy_score(y_va, pred_va_lgb)
    loso_scores_lgb.append(ba_lgb)

    oof_lgb[va_mask] = prob_va_lgb
    test_lgb += model_lgb.predict(X_test) / len(TRAIN_PIDS)

    # ── XGBoost ───────────────────────────────────────────────────────
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
    dval_xgb   = xgb.DMatrix(X_va, label=y_va)

    model_xgb = xgb.train(
        xgb_params,
        dtrain_xgb,
        num_boost_round=800,
        evals=[(dval_xgb, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False,
    )

    prob_va_xgb = model_xgb.predict(dval_xgb).reshape(-1, 5)
    pred_va_xgb = prob_va_xgb.argmax(axis=1)
    ba_xgb = balanced_accuracy_score(y_va, pred_va_xgb)
    loso_scores_xgb.append(ba_xgb)

    oof_xgb[va_mask] = prob_va_xgb
    test_xgb += model_xgb.predict(xgb.DMatrix(X_test)).reshape(-1, 5) / len(TRAIN_PIDS)

    print(f'  Subject {fold_pid} — LGB BA: {ba_lgb:.4f} | XGB BA: {ba_xgb:.4f}')

print(f'\nLOSO LGB  mean: {np.mean(loso_scores_lgb):.4f} ± {np.std(loso_scores_lgb):.4f}')
print(f'LOSO XGB  mean: {np.mean(loso_scores_xgb):.4f} ± {np.std(loso_scores_xgb):.4f}')

LOSO folds: 11 subjects
  Subject 01Z2 — LGB BA: 0.2333 | XGB BA: 0.1111
  Subject 70N8 — LGB BA: 0.1772 | XGB BA: 0.2213
  Subject 7PF3 — LGB BA: 0.1907 | XGB BA: 0.1786
  Subject CQ2G — LGB BA: 0.2546 | XGB BA: 0.2020
  Subject D1XP — LGB BA: 0.2403 | XGB BA: 0.2192
  Subject DT5C — LGB BA: 0.1915 | XGB BA: 0.2635
  Subject F1ZM — LGB BA: 0.1289 | XGB BA: 0.1426
  Subject LIUY — LGB BA: 0.0236 | XGB BA: 0.0797
  Subject SE4Q — LGB BA: 0.2500 | XGB BA: 0.2500
  Subject TPQI — LGB BA: 0.2185 | XGB BA: 0.1523
  Subject Y21H — LGB BA: 0.1498 | XGB BA: 0.2191

LOSO LGB  mean: 0.1871 ± 0.0647
LOSO XGB  mean: 0.1854 ± 0.0554


## Step 5 — OOF Ensemble & Final Blend

In [9]:
# Grid search best blend weight on OOF
best_ba, best_w = 0, 0.5

for w in np.arange(0.0, 1.01, 0.05):
    blend_oof = w * oof_lgb + (1 - w) * oof_xgb
    pred_oof  = blend_oof.argmax(axis=1)
    ba = balanced_accuracy_score(y_all, pred_oof)
    if ba > best_ba:
        best_ba = ba
        best_w  = w

print(f'Best OOF blend weight (LGB): {best_w:.2f} → OOF BA: {best_ba:.4f}')

# Final test predictions with best blend
test_blend = best_w * test_lgb + (1 - best_w) * test_xgb
test_pred  = test_blend.argmax(axis=1) + 1   # back to 1-5 scale

print(f'Test prediction distribution: {pd.Series(test_pred).value_counts().sort_index().to_dict()}')

Best OOF blend weight (LGB): 0.90 → OOF BA: 0.1782
Test prediction distribution: {1: 173, 2: 358, 3: 708, 4: 188, 5: 69}


In [10]:
submission = pd.DataFrame({
    'id':      test_feats['id'],
    'arousal': test_pred,
})

submission.to_csv('submission_v12.csv', index=False)
print('submission.csv saved.')
print(submission.head(10))
print(f'Shape: {submission.shape}')

submission.csv saved.
     id  arousal
0  2054        4
1  2055        3
2  2056        3
3  2057        3
4  2058        3
5  2059        3
6  2060        3
7  2061        3
8  2062        3
9  2063        3
Shape: (1496, 2)
